# 01 - Users Data Wrangling

Notebook ini memproses dataset `users.csv`.

Tabel `users` menyimpan informasi akun pengguna. Pada tahap ini, fokus utama bukan pada analisis perilaku, melainkan pada kerapian data identitas akun, validitas primary key, konsistensi email, dan kesiapan tabel ini sebagai referensi foreign key untuk tabel lain.

## 1. Import Library dan Setup Path

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 2. Load Dataset

In [2]:
# Membaca file CSV ke dalam dataframe.
users = pd.read_csv(RAW_DIR / "users.csv")
# Menampilkan beberapa baris awal untuk memahami bentuk data.
users.head()

,id,fullname,email,password_hash,profile_image,created_at,updated_at
0,1,Wahyu Saputra,wahyu.saputra1@example.com,96ace8d36b7a2c1423aa3886b70101fab26c0cd22733bd...,https://cdn.example.com/profiles/1.png,2025-08-19 01:00:00,2025-09-06 01:00:00
1,2,Rani Saputra,rani.saputra2@campusmail.id,54b54df2925579db43e616310494e29e6667dd72daa5a9...,https://cdn.example.com/profiles/2.png,2025-03-18 06:00:00,2025-06-02 06:00:00
2,3,Nadia Pratama,nadia.pratama3@example.com,5d2aea724fe1e5dd899fcb45f541f6b10c61c65ff5f292...,NaN,2025-08-24 16:00:00,2025-11-10 16:00:00
3,4,Alya Kurniawan,alya.kurniawan4@example.com,93523b0761950a989173d6870b82065494d82b9159d837...,https://cdn.example.com/profiles/4.png,2025-03-17 19:00:00,2025-04-15 19:00:00
4,5,Oki Permata,oki.permata5@studentmail.ac.id,d84def102e47bcb985144f28ca28e18e36a73cfd84e3ce...,https://cdn.example.com/profiles/5.png,2025-12-19 03:00:00,2026-03-19 03:00:00


## 3. Assessing Data

Assessing dilakukan untuk memahami struktur tabel, tipe data, nilai kosong, dan potensi duplicate. Pemeriksaan `users` difokuskan pada kolom `id`, `email`, `password_hash`, dan `profile_image`.

Tujuan dari tahap ini adalah memastikan tabel user dapat digunakan sebagai acuan relasi untuk tabel lain tanpa menghasilkan foreign key yang tidak valid.

In [3]:
# Menampilkan struktur kolom, tipe data, dan jumlah non-null.
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             300 non-null    int64
 1   fullname       300 non-null    str  
 2   email          300 non-null    str  
 3   password_hash  300 non-null    str  
 4   profile_image  219 non-null    str  
 5   created_at     300 non-null    str  
 6   updated_at     300 non-null    str  
dtypes: int64(1), str(6)
memory usage: 16.5 KB


In [4]:
# Menampilkan ringkasan statistik untuk kolom numerik dan kategorikal.
users.describe(include='all')

,id,fullname,email,password_hash,profile_image,created_at,updated_at
count,300.000000,300,300,300,219,300,300
unique,NaN,219,300,300,219,295,293
top,NaN,Iqbal Febriani,wahyu.saputra1@example.com,96ace8d36b7a2c1423aa3886b70101fab26c0cd22733bd...,https://cdn.example.com/profiles/1.png,2025-08-09 07:00:00,2025-09-06 01:00:00
freq,NaN,5,1,1,1,2,2
mean,150.500000,NaN,NaN,NaN,NaN,NaN,NaN
std,86.746758,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,75.750000,NaN,NaN,NaN,NaN,NaN,NaN
50%,150.500000,NaN,NaN,NaN,NaN,NaN,NaN
75%,225.250000,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
# Menghitung jumlah missing value pada setiap kolom.
users_missing = users.isna().sum()
# Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
users_duplicate_id = users["id"].duplicated().sum()
users_duplicate_email = users["email"].astype(str).str.strip().str.lower().duplicated().sum()
users_email_whitespace = users["email"].astype(str).str.match(r"^\s|\s$").sum()

print("Missing value:")
print(users_missing)

print("\nDuplicate id:", users_duplicate_id)
print("Duplicate email:", users_duplicate_email)
print("Email with whitespace:", users_email_whitespace)

Missing value:
id                0
fullname          0
email             0
password_hash     0
profile_image    81
created_at        0
updated_at        0
dtype: int64

Duplicate id: 0
Duplicate email: 0
Email with whitespace: 12


## Insight:

Berdasarkan hasil assessing, tabel `users` relatif sederhana dan tidak menunjukkan indikasi dirty data berat. Potensi masalah yang perlu diperhatikan adalah format teks, terutama whitespace pada email atau nama, serta kemungkinan duplicate pada `id` dan `email`.

Kolom `profile_image` dapat bernilai kosong karena tidak semua pengguna wajib memiliki foto profil. Oleh karena itu, nilai kosong pada kolom tersebut tidak diperlakukan sebagai kesalahan data.

Tindakan cleaning yang dilakukan adalah standardisasi format teks, validasi key, dan penghapusan duplicate jika ditemukan.

## 4. Cleaning Data

Cleaning dilakukan dengan langkah berikut:

1. Membersihkan whitespace pada kolom teks.
2. Menstandarkan `fullname` ke format title case.
3. Menstandarkan `email` ke lowercase.
4. Mengubah `id` ke numerik.
5. Mengubah `created_at` dan `updated_at` ke format datetime.
6. Menghapus baris yang tidak memiliki key atau email valid.
7. Menghapus duplicate berdasarkan `id` dan `email`.

In [6]:
# membuat salinan dataframe lalu menjalankan proses cleaning sesuai hasil assessing.
users_clean = users.copy()

# Membersihkan whitespace dan menstandarkan format teks.
users_clean["fullname"] = users_clean["fullname"].astype(str).str.strip().str.title()
users_clean["email"] = users_clean["email"].astype(str).str.strip().str.lower()
users_clean["password_hash"] = users_clean["password_hash"].astype(str).str.strip()
users_clean["profile_image"] = users_clean["profile_image"].fillna("").astype(str).str.strip()

# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
users_clean["id"] = pd.to_numeric(users_clean["id"], errors="coerce")
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
users_clean["created_at"] = pd.to_datetime(users_clean["created_at"], errors="coerce")
users_clean["updated_at"] = pd.to_datetime(users_clean["updated_at"], errors="coerce")

# Menghapus baris yang kehilangan kolom kunci atau informasi penting.
users_clean = users_clean.dropna(subset=["id", "email", "password_hash"])
# Menghapus duplicate sesuai subset key yang ditentukan.
users_clean = users_clean.drop_duplicates(subset=["id"], keep="last")
users_clean = users_clean.drop_duplicates(subset=["email"], keep="last")

users_clean["id"] = users_clean["id"].astype(int)
users_clean["created_at"] = users_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
users_clean["updated_at"] = users_clean["updated_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

users_clean = users_clean[
    ["id", "fullname", "email", "password_hash", "profile_image", "created_at", "updated_at"]
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
].sort_values("id")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
users_clean.head()

,id,fullname,email,password_hash,profile_image,created_at,updated_at
0,1,Wahyu Saputra,wahyu.saputra1@example.com,96ace8d36b7a2c1423aa3886b70101fab26c0cd22733bd...,https://cdn.example.com/profiles/1.png,2025-08-19 01:00:00,2025-09-06 01:00:00
1,2,Rani Saputra,rani.saputra2@campusmail.id,54b54df2925579db43e616310494e29e6667dd72daa5a9...,https://cdn.example.com/profiles/2.png,2025-03-18 06:00:00,2025-06-02 06:00:00
2,3,Nadia Pratama,nadia.pratama3@example.com,5d2aea724fe1e5dd899fcb45f541f6b10c61c65ff5f292...,,2025-08-24 16:00:00,2025-11-10 16:00:00
3,4,Alya Kurniawan,alya.kurniawan4@example.com,93523b0761950a989173d6870b82065494d82b9159d837...,https://cdn.example.com/profiles/4.png,2025-03-17 19:00:00,2025-04-15 19:00:00
4,5,Oki Permata,oki.permata5@studentmail.ac.id,d84def102e47bcb985144f28ca28e18e36a73cfd84e3ce...,https://cdn.example.com/profiles/5.png,2025-12-19 03:00:00,2026-03-19 03:00:00


## 5. Validation dan Save Output

In [7]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
# Membuat dataframe validasi untuk mendokumentasikan hasil pengecekan kualitas data.
validation = pd.DataFrame([
    {"rule": "users.id unique", "passed": users_clean["id"].is_unique},
    {"rule": "users.email unique", "passed": users_clean["email"].is_unique},
    {"rule": "password_hash not null", "passed": users_clean["password_hash"].notna().all()},
])

validation

,rule,passed
0,users.id unique,True
1,users.email unique,True
2,password_hash not null,True


In [8]:
# menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
users_clean.to_csv(PROCESSED_DIR / "users_clean.csv", index=False)
validation.to_csv(REPORT_DIR / "users_validation.csv", index=False)

print("Saved:", PROCESSED_DIR / "users_clean.csv")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\users_clean.csv
